In [1]:
# Cell 1 — imports
import sys
sys.path.append(".")

import pandas as pd
from modelling_utils import (
    SpatialSplitter, ClassBalancer, RFTrainer, ModelPersister,
    FEATURE_COLUMNS, BINARY_MODEL_PATH,
)

TRAINING_CSV_PATH = "../data/output/nairobi_training_pixels_clean.csv"

In [2]:
# Cell 2 — load the cleaned dataset
df = pd.read_csv(TRAINING_CSV_PATH)
print(df.shape)
print(df["built_up"].value_counts(normalize=True))

(11998745, 12)
built_up
0    0.921908
1    0.078092
Name: proportion, dtype: float64


In [3]:
# Cell 3 — spatial train/test split (by tile, not by row)
splitter = SpatialSplitter(test_size=0.2, random_state=42)
train_df, test_df = splitter.split(df, group_col="tile_id")

print(f"Train: {len(train_df):,} rows across {train_df['tile_id'].nunique()} tiles")
print(f"Test:  {len(test_df):,} rows across {test_df['tile_id'].nunique()} tiles")

# Confirm no tile appears in both splits
overlap = set(train_df["tile_id"]) & set(test_df["tile_id"])
print(f"Tiles in both splits (should be empty): {overlap}")

Train: 9,499,083 rows across 38 tiles
Test:  2,499,662 rows across 10 tiles
Tiles in both splits (should be empty): set()


In [4]:
# Cell 4 — optional: subsample the training set's majority class
# (class_weight='balanced' in RFTrainer already helps too — you can use
# either approach alone, or both together, this is just giving you the option)
balancer = ClassBalancer(majority_ratio=8.0, random_state=42)
train_df_balanced = balancer.subsample(train_df, label_col="built_up")

print(f"Balanced training set: {len(train_df_balanced):,} rows")
print(train_df_balanced["built_up"].value_counts(normalize=True))

Balanced training set: 7,325,289 rows
built_up
0    0.888889
1    0.111111
Name: proportion, dtype: float64


In [9]:
# Cell 5 — train the binary RF
trainer = RFTrainer(
    feature_columns=FEATURE_COLUMNS,
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
)
trainer.fit(train_df_balanced, label_col="built_up")

In [10]:
# Cell 6 — evaluate on the held-out test tiles
results = trainer.evaluate(test_df, label_col="built_up")

              precision    recall  f1-score   support

           0       0.97      0.96      0.97   2376572
           1       0.38      0.43      0.41    123090

    accuracy                           0.94   2499662
   macro avg       0.68      0.70      0.69   2499662
weighted avg       0.94      0.94      0.94   2499662

Confusion matrix:
 [[2291240   85332]
 [  69836   53254]]
Accuracy: 0.9379
F1 (weighted): 0.9397


In [11]:
# Cell 7 — feature importance (which columns the RF actually relied on)
importance_df = trainer.feature_importance()
importance_df

,feature,importance
0,ndvi,0.196601
1,ndwi,0.172539
2,B2,0.159748
3,B8,0.133937
4,B4,0.118666
5,distance_to_river_m,0.116681
6,B3,0.101828
7,ndbi,0.000000


In [12]:
# Cell 8 — pickle the trained model
ModelPersister.save(trainer.model, str(BINARY_MODEL_PATH))

Model saved to models\rf_built_up_binary.pkl


In [5]:
# Cell 9 — sanity check: reload and confirm it predicts the same way
reloaded_model = ModelPersister.load(str(BINARY_MODEL_PATH))
sample = test_df[FEATURE_COLUMNS].iloc[:5]
print(reloaded_model.predict(sample))
#print(trainer.model.predict(sample))  # should match exactly

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- ndbi
